# 🦺 PPE Detection System — Google Colab Training Pipeline
**Antigravity Edge Compute Platform | Computer Vision Engineer Assignment**

---

### 🗺️ Pipeline Overview
| Step | Cell | Description |
|------|------|-------------|
| 1 | Setup | Verify GPU, install dependencies |
| 2 | Dataset | Upload & configure dataset from your local machine |
| 3 | Train | Fine-tune YOLOv8n for 50 epochs |
| 4 | Validate | Extract mAP50 / mAP50-95 metrics |
| 5 | Export | Convert to ONNX FP16 with memory audit |
| 6 | Download | Save weights to your local machine |

> ⚡ **Make sure GPU is enabled:** `Runtime → Change runtime type → T4 GPU`

---
## 🔧 Step 1 — Environment Setup & GPU Verification

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Verify GPU and install all required dependencies
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

# --- Check GPU ---
print('=' * 60)
print('  GPU VERIFICATION')
print('=' * 60)
!nvidia-smi

# --- Python version ---
print(f'\nPython: {sys.version}')

# --- Install / upgrade dependencies ---
print('\n' + '=' * 60)
print('  INSTALLING DEPENDENCIES')
print('=' * 60)
!pip install -q ultralytics onnx onnxruntime opencv-python-headless

# --- Verify torch + CUDA ---
import torch
print(f'\nPyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print(f'\n✅ GPU Ready: {torch.cuda.is_available()}')

---
## 📂 Step 2 — Dataset Upload

**You have 2 options:**
- **Option A** *(Recommended)* — Upload your dataset zip from your local machine
- **Option B** — Mount Google Drive if your dataset is already there

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2A: OPTION A — Upload dataset zip from your Windows machine
#
# INSTRUCTIONS:
#   1. On your Windows machine, zip your dataset folder:
#      Right-click the 'dataset' folder → Send to → Compressed (zipped) folder
#      This creates 'dataset.zip'
#   2. Run this cell — a file picker will appear
#   3. Select your 'dataset.zip' file
# ─────────────────────────────────────────────────────────────────────────────
import os
from google.colab import files

print('📂 Please select your dataset.zip file from your local machine...')
uploaded = files.upload()  # Opens file picker dialog

# Get the uploaded filename
zip_name = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_name}')

# Extract to /content/dataset/
!mkdir -p /content/dataset
!unzip -q "{zip_name}" -d /content/dataset/

# List contents to verify
print('\n📁 Extracted contents:')
!find /content/dataset -maxdepth 3 | head -30

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2B: OPTION B — Mount Google Drive (skip if using Option A)
#
# Use this if you've already uploaded your dataset to Google Drive.
# Update DRIVE_DATASET_PATH to point to your dataset folder in Drive.
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Update this path to match where your dataset folder lives in Drive:
DRIVE_DATASET_PATH = '/content/drive/MyDrive/dataset'  # <-- CHANGE THIS

import os
if os.path.exists(DRIVE_DATASET_PATH):
    !ln -s "{DRIVE_DATASET_PATH}" /content/dataset
    print(f'✅ Dataset linked from Drive: {DRIVE_DATASET_PATH}')
else:
    print(f'❌ Path not found: {DRIVE_DATASET_PATH}')
    print('   Please update DRIVE_DATASET_PATH above.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2C: Auto-detect dataset structure & generate data.yaml
#
# This cell auto-finds train/valid/test splits and writes a corrected
# data.yaml with absolute Colab paths. Handles both flat and nested zips.
# ─────────────────────────────────────────────────────────────────────────────
import os
import yaml

BASE = '/content/dataset'

def find_split(base, split_name):
    """Find the images directory for a given split name."""
    candidates = [
        f'{base}/{split_name}/images',
        f'{base}/dataset/{split_name}/images',
        f'{base}/{split_name}',
    ]
    for c in candidates:
        if os.path.isdir(c):
            imgs = [f for f in os.listdir(c) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                return c, len(imgs)
    return None, 0

# --- Detect splits ---
print('🔍 Detecting dataset splits...')
train_path, train_n = find_split(BASE, 'train')
valid_path, valid_n = find_split(BASE, 'valid')
test_path,  test_n  = find_split(BASE, 'test')

print(f'  train : {train_path}  ({train_n} images)')
print(f'  valid : {valid_path}  ({valid_n} images)')
print(f'  test  : {test_path}   ({test_n} images)')

if not train_path or not valid_path:
    raise RuntimeError('❌ Could not find train or valid splits. Check your zip structure.')

# --- Write corrected data.yaml ---
data_yaml_content = {
    'train': train_path,
    'val'  : valid_path,
    'test' : test_path if test_path else valid_path,
    'nc'   : 4,
    'names': ['Gloves', 'Vest', 'helmet', 'person']
}

YAML_PATH = '/content/data.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f'\n✅ data.yaml written to: {YAML_PATH}')
print('\nContents:')
!cat /content/data.yaml

---
## 🏋️ Step 3 — Train YOLOv8n (50 Epochs, GPU)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3: Training Pipeline
#
# Trains YOLOv8n on the PPE dataset with:
#   - 50 epochs, imgsz=640, AutoBatch on GPU
#   - Early stopping (patience=10)
#   - Training curves + confusion matrix plots
# Expected time on Colab T4 GPU: ~10-20 minutes
# ─────────────────────────────────────────────────────────────────────────────
import time
from ultralytics import YOLO

# Configuration
PROJECT_DIR = '/content/runs'
RUN_NAME    = 'ppe_yolov8n_training'

print('=' * 65)
print('  PPE DETECTION — YOLOv8n TRAINING  |  Antigravity Platform')
print('=' * 65)
print(f'  Model    : yolov8n.pt (pretrained baseline)')
print(f'  Dataset  : {YAML_PATH}')
print(f'  Epochs   : 50')
print(f'  ImgSize  : 640 × 640')
print(f'  Batch    : -1 (AutoBatch)')
print(f'  Device   : GPU (cuda:0)')
print('=' * 65 + '\n')

# --- Load baseline model ---
model = YOLO('yolov8n.pt')

# --- Start training ---
t0 = time.time()

results = model.train(
    data      = YAML_PATH,
    imgsz     = 640,
    epochs    = 50,
    batch     = -1,          # AutoBatch: estimates optimal batch for GPU memory
    device    = 0,           # cuda:0 (T4 GPU on Colab)
    project   = PROJECT_DIR,
    name      = RUN_NAME,
    exist_ok  = True,
    patience  = 10,          # Early stopping
    plots     = True,        # Training curves + confusion matrix
    save      = True,
    workers   = 2,           # Colab DataLoader workers
    verbose   = True,
)

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')

# --- Locate weights ---
BEST_PT = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'
LAST_PT = f'{PROJECT_DIR}/{RUN_NAME}/weights/last.pt'
print(f'  Best weights : {BEST_PT}')
print(f'  Last weights : {LAST_PT}')

---
## 📊 Step 4 — Validation & mAP Metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4: Post-training validation using best.pt
# Extracts mAP50, mAP50-95, Precision, Recall, and per-class AP
# ─────────────────────────────────────────────────────────────────────────────
from ultralytics import YOLO
import os

BEST_PT  = f'/content/runs/ppe_yolov8n_training/weights/best.pt'
YAML_PATH = '/content/data.yaml'

print('Loading best checkpoint for validation...')
val_model = YOLO(BEST_PT)

val_results = val_model.val(
    data    = YAML_PATH,
    imgsz   = 640,
    device  = 0,
    split   = 'val',
    verbose = True,
)

# --- Extract core metrics ---
map50    = val_results.box.map50
map50_95 = val_results.box.map
precision= val_results.box.mp
recall   = val_results.box.mr

CLASS_NAMES = ['Gloves', 'Vest', 'helmet', 'person']

print('\n' + '═' * 55)
print('        VALIDATION RESULTS — PERFORMANCE SUMMARY')
print('═' * 55)
print(f'  mAP @ 0.50       : {map50:.4f}')
print(f'  mAP @ 0.50:0.95  : {map50_95:.4f}')
print(f'  Mean Precision   : {precision:.4f}')
print(f'  Mean Recall      : {recall:.4f}')
print('─' * 55)

# Per-class AP
if hasattr(val_results.box, 'ap50') and val_results.box.ap50 is not None:
    print('  PER-CLASS AP @ 0.50:')
    for i, ap in enumerate(val_results.box.ap50):
        name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f'class_{i}'
        bar  = '█' * int(ap * 20)
        print(f'    {name:<10} : {ap:.4f}  {bar}')

print('═' * 55)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4B: Visualize training curves and confusion matrix
# ─────────────────────────────────────────────────────────────────────────────
import glob
from IPython.display import Image, display

RUN_DIR = '/content/runs/ppe_yolov8n_training'

# Plot files generated by Ultralytics
plot_files = {
    'Training Curves'      : f'{RUN_DIR}/results.png',
    'Confusion Matrix'     : f'{RUN_DIR}/confusion_matrix.png',
    'Precision-Recall Curve': f'{RUN_DIR}/PR_curve.png',
    'F1 Curve'             : f'{RUN_DIR}/F1_curve.png',
    'Label Distribution'   : f'{RUN_DIR}/labels.jpg',
}

for title, path in plot_files.items():
    if os.path.exists(path):
        print(f'\n📊 {title}:')
        display(Image(filename=path, width=800))
    else:
        print(f'  ⚠️  {title} not found at: {path}')

---
## 📦 Step 5 — ONNX Export with FP16 Quantization & Memory Audit

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5: Export trained model to ONNX FP16 format
#
# Parameters:
#   format  = 'onnx'   — interoperable ONNX serialization
#   half    = True     — FP16 half-precision weight quantization
#   opset   = 12       — ONNX opset 12 (broad compatibility)
#   simplify= True     — run onnx-simplifier to optimize graph
# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
from ultralytics import YOLO
import onnx

BEST_PT   = '/content/runs/ppe_yolov8n_training/weights/best.pt'
EXPORT_DIR = '/content/models'
os.makedirs(EXPORT_DIR, exist_ok=True)

# --- Load model ---
print('Loading best.pt for ONNX export...')
model = YOLO(BEST_PT)

fp32_size = os.path.getsize(BEST_PT) / (1024 * 1024)
print(f'  FP32 source size : {fp32_size:.2f} MB')

# --- Export ---
print('\nExporting to ONNX FP16 (opset 12)...')
export_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    half     = True,      # FP16 quantization
    opset    = 12,
    simplify = True,
    dynamic  = False,
)

onnx_path = Path(export_path)

# --- ONNX Graph Validation ---
print('\nValidating ONNX graph...')
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print('  ✅ ONNX graph validation PASSED')

# --- Copy to models dir ---
import shutil
deploy_path = f'{EXPORT_DIR}/ppe_yolov8n_fp16.onnx'
shutil.copy2(str(onnx_path), deploy_path)

fp16_size  = os.path.getsize(deploy_path) / (1024 * 1024)
reduction  = fp32_size - fp16_size
ratio      = (reduction / fp32_size) * 100

# --- Memory Audit ---
print('\n' + '═' * 55)
print('        MEMORY AUDIT — STORAGE REDUCTION ANALYSIS')
print('═' * 55)
print(f'  Original  FP32 (.pt)    : {fp32_size:>8.2f} MB')
print(f'  Quantized FP16 (.onnx)  : {fp16_size:>8.2f} MB')
print(f'  ──────────────────────────────────────────────')
print(f'  Absolute Reduction      : {reduction:>8.2f} MB')
print(f'  Compression Ratio       : {ratio:>8.1f} %')
print('═' * 55)
print(f'\n✅ FP16 model saved to: {deploy_path}')

---
## 💾 Step 6 — Download All Artifacts to Your Windows Machine

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6A: Download best.pt (FP32 PyTorch weights)
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import files
print('📥 Downloading best.pt (FP32 weights)...')
files.download('/content/runs/ppe_yolov8n_training/weights/best.pt')
print('✅ best.pt downloaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6B: Download ppe_yolov8n_fp16.onnx (quantized ONNX model)
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import files
print('📥 Downloading ppe_yolov8n_fp16.onnx (FP16 quantized)...')
files.download('/content/models/ppe_yolov8n_fp16.onnx')
print('✅ ppe_yolov8n_fp16.onnx downloaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6C: Zip and download all training artifacts (curves, plots, weights)
# ─────────────────────────────────────────────────────────────────────────────
import shutil
from google.colab import files

print('📦 Zipping training run artifacts...')
shutil.make_archive(
    '/content/ppe_training_results',
    'zip',
    '/content/runs/ppe_yolov8n_training'
)

size_mb = os.path.getsize('/content/ppe_training_results.zip') / (1024*1024)
print(f'  Archive size: {size_mb:.1f} MB')
print('\n📥 Downloading full results archive...')
files.download('/content/ppe_training_results.zip')
print('✅ ppe_training_results.zip downloaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6D: OPTIONAL — Save everything to Google Drive (avoids re-uploading)
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=True)

DRIVE_SAVE_PATH = '/content/drive/MyDrive/PPE_Detection_Results'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# Copy key files
for src, dst_name in [
    ('/content/runs/ppe_yolov8n_training/weights/best.pt',  'best.pt'),
    ('/content/runs/ppe_yolov8n_training/weights/last.pt',  'last.pt'),
    ('/content/models/ppe_yolov8n_fp16.onnx',              'ppe_yolov8n_fp16.onnx'),
    ('/content/ppe_training_results.zip',                   'ppe_training_results.zip'),
]:
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_SAVE_PATH}/{dst_name}')
        print(f'  ✅ Saved: {dst_name}')
    else:
        print(f'  ⚠️  Not found: {src}')

print(f'\n✅ All artifacts saved to Google Drive: {DRIVE_SAVE_PATH}')

---
## ✅ Done! Next Steps on Your Windows Machine

After downloading the files:

1. **Place files** into your project folder:
   ```
   Computer Vision Engineer_Assignment/
   ├── runs/ppe_yolov8n_training/weights/
   │   └── best.pt          ← put here
   └── models/
       └── ppe_yolov8n_fp16.onnx  ← put here
   ```

2. **Run live inference** (already set up on your machine):
   ```bash
   python live_inference.py
   ```

3. **Or re-run the ONNX export locally** (optional):
   ```bash
   python export_onnx.py
   ```

---
*PPE Detection System | Antigravity Edge Compute Platform | 2026*